In [7]:
#Ishaan
#Labels
import math
import pandas as pd
import numpy as np
def soft_labels_generation(ratings):
    soft_list = []
    for i in ratings: 
        temp_list = [int(x) for x in i.split(";")]
        n = len(temp_list)
        soft_list.append([temp_list.count(1)/n, temp_list.count(2)/n,temp_list.count(3)/n,temp_list.count(4)/n,temp_list.count(5)/n])
    return np.array(soft_list) # shape (num_nodules, 5)
    
def hard_labels_generation(ratings):
    hard_list = []
    for i in ratings:
        temp_list = sorted(int(x) for x in i.split(";")) # median, not mean: mean gives floats (regression); median keeps 5 classes
        n = len(temp_list)
        median = temp_list[n//2] if n % 2 else (temp_list[n//2 - 1] + temp_list[n//2]) / 2
        hard_list.append(math.floor(median + 0.5))     #not round(), we dont want to use banker's rounding
    return np.array(hard_list) # shape (num_nodules,), values 1..5
    
# 4;5;5;5 ratings = [4, 5, 5, 5]  →  hard = 5,  soft = [0, 0, 0, 0.25, 0.75]
        
labels_bb_a = "data/labels_bb_a.csv"
df = pd.read_csv(labels_bb_a)
nodules = df.drop_duplicates(["patient_id", "series_instance_uid", "merged_nodule_id"]).copy()
nodules = nodules[nodules["num_readers"] >= 2].reset_index(drop=True)
soft_labels = soft_labels_generation(nodules["ratings"])
hard_labels = hard_labels_generation(nodules["ratings"])

[[0.   0.   0.   0.25 0.75]
 [0.   0.   0.   0.5  0.5 ]
 [0.   0.   0.25 0.25 0.5 ]
 ...
 [0.   0.   0.25 0.   0.75]
 [0.   0.   0.5  0.5  0.  ]
 [0.   1.   0.   0.   0.  ]]


16172 slice rows -> 1885 nodules (num_readers >= 2)
hard label counts:
 hard
1    206
2    288
3    898
4    316
5    177
Name: count, dtype: int64
spread counts:
 spread
0    409
1    718
2    552
3    179
4     27
Name: count, dtype: int64
    patient_id                                              series_instance_uid  merged_nodule_id  num_readers ratings  hard  soft_1  soft_2  soft_3  soft_4  soft_5  spread                                                    anchor_sop_id  anchor_z  x_centre  y_centre  diameter_px  n_slices
LIDC-IDRI-0001 1.3.6.1.4.1.14519.5.2.1.6279.6001.179049373636438705059720603192                 1            4 5;5;5;4     5     0.0     0.0     0.0    0.25    0.75       1 1.3.6.1.4.1.14519.5.2.1.6279.6001.824843590991776411530080688091    -117.5   316.375    363.25         43.5         9
saved data/nodules_labels.csv
